# Decorators

Split from `01_decorators_and_context_managers.ipynb` to keep this topic self-contained.

**Navigation:** [Topic overview](./01_decorators_and_context_managers.ipynb) · [Next: Context Managers](./02_context_managers.ipynb)


## Why decorators matter in bioinformatics

Decorators let you add behavior to functions without changing their body: time how long sequence alignment takes, validate that a DNA string contains only ATGC, cache expensive translation lookups, or retry a flaky NCBI API call. These cross-cutting concerns would otherwise clutter every function with repetitive boilerplate.

## How to use this notebook

Run cells from top to bottom on the first pass — later cells depend on functions and variables defined earlier. Once you have run through the notebook, feel free to modify parameters and re-run individual sections.

Each section has runnable examples first, followed by exercises. Try the exercise before looking at the solution cell below it.

## Complicated moments explained

**1. A decorator is just a callable that takes a callable**
`@timer` above `def analyze(seq):` is exactly `analyze = timer(analyze)`. It replaces the function with its decorated version at definition time.

**2. Always use `functools.wraps`**
Without `@functools.wraps(func)`, the wrapper has `__name__ = 'wrapper'` and no docstring. This breaks `help()`, logging, and any tool that inspects function metadata.

**3. Decorator factories need three levels**
When the decorator takes arguments (`@validate_sequence('ATGC')`), add one extra level:
```python
def validate_sequence(valid_chars):   # factory — takes the argument
    def decorator(func):              # actual decorator — takes the function
        @functools.wraps(func)
        def wrapper(*args, **kwargs): # called at runtime
            ...
        return wrapper
    return decorator
```

**4. Stacking decorators applies bottom-up**
`@A @B def f():` is `f = A(B(f))`. The decorator closest to the function (`B`) is applied first.

**5. `lru_cache` requires hashable arguments**
`@lru_cache` caches by argument value. All arguments must be hashable — lists and dicts cannot be cached directly.

In [ ]:
def gc_content(seq):
    """Calculate GC content of a DNA sequence."""
    seq = seq.upper()
    return (seq.count('G') + seq.count('C')) / len(seq) * 100

def at_content(seq):
    """Calculate AT content of a DNA sequence."""
    seq = seq.upper()
    return (seq.count('A') + seq.count('T')) / len(seq) * 100

# Functions are objects -- assign to variable, put in list
analyzers = [gc_content, at_content]

test_seq = "ATGCGATCGATCGTAGC"
for func in analyzers:
    print(f"{func.__name__}: {func(test_seq):.1f}%")

In [ ]:
# Higher-order function: takes a function as argument
def apply_to_sequences(sequences, analysis_func):
    """Apply an analysis function to multiple sequences."""
    return {name: analysis_func(seq) for name, seq in sequences.items()}

seqs = {
    "BRCA1": "ATGGATTTCGATCGATCGTAGC",
    "TP53":  "ATGGAGGAGCCGCAGTCAGATC",
    "MYC":   "GGCCAATTGGCCAATTGGCC",
}

gc_results = apply_to_sequences(seqs, gc_content)
for gene, gc in gc_results.items():
    print(f"{gene}: {gc:.1f}% GC")

## 2. Closures: Functions That Remember

A closure is an inner function that captures variables from its enclosing scope.
This lets you create specialized functions at runtime.

In [ ]:
def make_motif_counter(motif):
    """Return a function that counts a specific motif in any sequence."""
    motif = motif.upper()

    def counter(sequence):
        return sequence.upper().count(motif)

    return counter

# Create specialized counters
count_cpg = make_motif_counter("CG")
count_tata = make_motif_counter("TATA")

promoter = "GCTATAAAAGGCGCGCGTATAATCGCG"
print(f"CG dinucleotides: {count_cpg(promoter)}")
print(f"TATA boxes: {count_tata(promoter)}")

# The inner function 'remembers' the motif even after make_motif_counter returns
print(f"\nClosure variables: {count_cpg.__closure__[0].cell_contents}")

## 3. Decorators: The Basics

A decorator is a function that takes a function and returns a modified version of it.

```
DECORATOR PATTERN
+-------------------------------------------+
|  @decorator        is equivalent to        |
|  def func():  -->  func = decorator(func)  |
+-------------------------------------------+
```

In [ ]:
import functools


def log_call(func):
    """Decorator that logs when a function is called."""
    @functools.wraps(func)  # preserves __name__, __doc__, etc.
    def wrapper(*args, **kwargs):
        print(f"Calling {func.__name__}...")
        result = func(*args, **kwargs)
        print(f"{func.__name__} returned {result}")
        return result
    return wrapper


@log_call
def count_nucleotides(seq):
    """Count each nucleotide in a DNA sequence."""
    seq = seq.upper()
    return {base: seq.count(base) for base in 'ATGC'}


result = count_nucleotides("ATGCGATCGATCG")
print(f"\nResult: {result}")

# functools.wraps preserves the original function's identity
print(f"Function name: {count_nucleotides.__name__}")
print(f"Docstring: {count_nucleotides.__doc__}")

### Why `functools.wraps` Matters

Without `@functools.wraps(func)`, the decorated function would have:
- `__name__` = `"wrapper"` instead of the original name
- `__doc__` = the wrapper's docstring instead of the original

This breaks introspection, help(), and debugging. Always use `functools.wraps`.

## 4. Timing Decorator: Benchmarking Bioinformatics Algorithms

In [ ]:
import time


def timer(func):
    """Measure and print execution time of a function."""
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        elapsed = time.perf_counter() - start
        print(f"[timer] {func.__name__}: {elapsed:.4f} seconds")
        return result
    return wrapper

In [ ]:
import random


@timer
def gc_content_loop(seq):
    """GC content via explicit loop."""
    gc_count = 0
    for base in seq:
        if base in 'GCgc':
            gc_count += 1
    return gc_count / len(seq) * 100


@timer
def gc_content_count(seq):
    """GC content via str.count()."""
    seq = seq.upper()
    gc = seq.count('G') + seq.count('C')
    return gc / len(seq) * 100


# Generate a 1 million bp test sequence
big_seq = ''.join(random.choices('ATGC', k=1_000_000))

print("Comparing GC content methods on 1M bp:")
r1 = gc_content_loop(big_seq)
r2 = gc_content_count(big_seq)
print(f"Both give: {r1:.2f}% vs {r2:.2f}%")

## 5. Sequence Validation Decorator

In [ ]:
def validate_sequence(valid_chars, seq_type="DNA"):
    """Decorator factory: validate that the first argument is a valid sequence."""
    valid_set = set(valid_chars.upper())

    def decorator(func):
        @functools.wraps(func)
        def wrapper(seq, *args, **kwargs):
            invalid = set(seq.upper()) - valid_set
            if invalid:
                raise ValueError(
                    f"Invalid {seq_type} characters {invalid} in input to {func.__name__}()"
                )
            return func(seq, *args, **kwargs)
        return wrapper
    return decorator


@validate_sequence('ATGC', seq_type='DNA')
def complement(seq):
    """Return the DNA complement."""
    comp_map = {'A': 'T', 'T': 'A', 'G': 'C', 'C': 'G'}
    return ''.join(comp_map[b] for b in seq.upper())


@validate_sequence('ACDEFGHIKLMNPQRSTVWY', seq_type='protein')
def protein_mass(seq):
    """Estimate protein mass (rough, 110 Da per residue)."""
    return len(seq) * 110


# Valid input
print(complement("ATGCGATC"))

# Invalid DNA
try:
    complement("ATGXYZ")
except ValueError as e:
    print(f"Caught: {e}")

# Invalid protein
try:
    protein_mass("MVLS123")
except ValueError as e:
    print(f"Caught: {e}")

## 6. Decorators with Arguments

When a decorator needs parameters (like `@validate_sequence('ATGC')`),
you need a three-level nesting: a factory that returns the actual decorator.

In [ ]:
def repeat(n=3):
    """Decorator factory: run the function n times and return all results."""
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            return [func(*args, **kwargs) for _ in range(n)]
        return wrapper
    return decorator


@repeat(n=5)
def random_gc_sample(seq, sample_size=100):
    """GC content of a random subsequence."""
    start = random.randint(0, len(seq) - sample_size)
    sample = seq[start:start + sample_size].upper()
    gc = (sample.count('G') + sample.count('C')) / len(sample) * 100
    return round(gc, 1)


gc_samples = random_gc_sample(big_seq, sample_size=500)
print(f"GC content from 5 random 500-bp windows: {gc_samples}")
print(f"Mean: {sum(gc_samples) / len(gc_samples):.1f}%")

## 7. Memoization: Caching Expensive Results

In [ ]:
def memoize(func):
    """Cache function results for repeated calls with the same arguments."""
    cache = {}

    @functools.wraps(func)
    def wrapper(*args):
        if args not in cache:
            cache[args] = func(*args)
        return cache[args]

    wrapper.cache = cache
    wrapper.clear_cache = cache.clear
    return wrapper


# Recursive Needleman-Wunsch alignment score (simplified)
@memoize
def align_score(seq1, seq2, match=1, mismatch=-1, gap=-2):
    """Recursive pairwise alignment score with memoization."""
    if not seq1:
        return len(seq2) * gap
    if not seq2:
        return len(seq1) * gap

    score = match if seq1[-1] == seq2[-1] else mismatch

    return max(
        align_score(seq1[:-1], seq2[:-1]) + score,
        align_score(seq1[:-1], seq2) + gap,
        align_score(seq1, seq2[:-1]) + gap,
    )


s1 = "ACGTACGT"
s2 = "ACGTAGCT"
print(f"Alignment score for '{s1}' vs '{s2}': {align_score(s1, s2)}")
print(f"Cache entries: {len(align_score.cache)}")

In [ ]:
# Python's built-in lru_cache is more efficient and has a size limit
from functools import lru_cache


@lru_cache(maxsize=128)
def translate_codon(codon):
    """Translate a single codon to amino acid (cached)."""
    table = {
        'TTT': 'F', 'TTC': 'F', 'TTA': 'L', 'TTG': 'L',
        'TCT': 'S', 'TCC': 'S', 'TCA': 'S', 'TCG': 'S',
        'TAT': 'Y', 'TAC': 'Y', 'TAA': '*', 'TAG': '*',
        'TGT': 'C', 'TGC': 'C', 'TGA': '*', 'TGG': 'W',
        'CTT': 'L', 'CTC': 'L', 'CTA': 'L', 'CTG': 'L',
        'CCT': 'P', 'CCC': 'P', 'CCA': 'P', 'CCG': 'P',
        'CAT': 'H', 'CAC': 'H', 'CAA': 'Q', 'CAG': 'Q',
        'CGT': 'R', 'CGC': 'R', 'CGA': 'R', 'CGG': 'R',
        'ATT': 'I', 'ATC': 'I', 'ATA': 'I', 'ATG': 'M',
        'ACT': 'T', 'ACC': 'T', 'ACA': 'T', 'ACG': 'T',
        'AAT': 'N', 'AAC': 'N', 'AAA': 'K', 'AAG': 'K',
        'AGT': 'S', 'AGC': 'S', 'AGA': 'R', 'AGG': 'R',
        'GTT': 'V', 'GTC': 'V', 'GTA': 'V', 'GTG': 'V',
        'GCT': 'A', 'GCC': 'A', 'GCA': 'A', 'GCG': 'A',
        'GAT': 'D', 'GAC': 'D', 'GAA': 'E', 'GAG': 'E',
        'GGT': 'G', 'GGC': 'G', 'GGA': 'G', 'GGG': 'G',
    }
    return table.get(codon.upper(), 'X')


@timer
def translate_sequence(dna):
    """Translate a DNA sequence using the cached codon lookup."""
    protein = []
    for i in range(0, len(dna) - 2, 3):
        aa = translate_codon(dna[i:i+3])
        if aa == '*':
            break
        protein.append(aa)
    return ''.join(protein)


# Translate a long repeated sequence to see caching in action
long_dna = "ATGGCTGCTTAG" * 1000
protein = translate_sequence(long_dna)
print(f"Protein (first 30 aa): {protein[:30]}...")
print(f"Cache info: {translate_codon.cache_info()}")

## 8. Stacking Multiple Decorators

Decorators are applied bottom-up (closest to the function first).

In [ ]:
@timer
@validate_sequence('ATGC', seq_type='DNA')
def analyze_sequence(seq):
    """Full analysis of a DNA sequence."""
    seq = seq.upper()
    return {
        'length': len(seq),
        'gc_content': (seq.count('G') + seq.count('C')) / len(seq) * 100,
        'starts_with_atg': seq.startswith('ATG'),
        'a': seq.count('A'),
        't': seq.count('T'),
        'g': seq.count('G'),
        'c': seq.count('C'),
    }


# Valid sequence -- gets validated then timed
result = analyze_sequence("ATGCGATCGATCGATCGATCG")
print(f"Analysis: {result}")

# Invalid -- validation catches it before timing even starts
try:
    analyze_sequence("ATGXYZ")
except ValueError as e:
    print(f"\nCaught: {e}")

---

## 9. Context Managers: Safe Resource Handling

Context managers ensure resources (files, connections, locks) are properly
acquired and released, even when exceptions occur.

```
CONTEXT MANAGER LIFECYCLE
+-----------------------------------+
|  with ctx as resource:            |
|      |                            |
|      v                            |
|  __enter__() -> acquire resource  |
|      |                            |
|      v                            |
|  (your code runs here)            |
|      |                            |
|      v                            |
|  __exit__() -> release resource   |
+-----------------------------------+
```

In [ ]:
class FastaWriter:
    """Context manager for writing FASTA files."""

    def __init__(self, filename, line_width=80):
        self.filename = filename
        self.line_width = line_width
        self.file = None
        self.record_count = 0

    def __enter__(self):
        self.file = open(self.filename, 'w')
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        if self.file:
            self.file.close()
        if exc_type is not None:
            print(f"Error occurred while writing: {exc_val}")
        print(f"Wrote {self.record_count} records to {self.filename}")
        return False  # do not suppress exceptions

    def write_record(self, seq_id, sequence, description=""):
        """Write a single FASTA record with line wrapping."""
        header = f">{seq_id}" + (f" {description}" if description else "")
        self.file.write(header + "\n")
        for i in range(0, len(sequence), self.line_width):
            self.file.write(sequence[i:i + self.line_width] + "\n")
        self.record_count += 1


# Write sequences safely -- file is always closed
with FastaWriter("output_test.fasta") as writer:
    writer.write_record("BRCA1", "ATGGATTTCGATCG" * 10, "breast cancer gene")
    writer.write_record("TP53", "ATGGAGGAGCCGCAG" * 8, "tumor protein p53")
    writer.write_record("EGFR", "ATGCGACCCTCCGGG" * 12, "epidermal growth factor receptor")

# Verify
with open("output_test.fasta") as f:
    print(f.read()[:300] + "...")

## 10. Context Managers with `contextlib`

For simple context managers, the `@contextmanager` decorator from `contextlib`
avoids the need for a full class.

In [ ]:
from contextlib import contextmanager
import os


@contextmanager
def timed_section(name):
    """Time a block of code."""
    start = time.perf_counter()
    try:
        yield
    finally:
        elapsed = time.perf_counter() - start
        print(f"[{name}] {elapsed:.4f}s")


with timed_section("GC content of 1M bp"):
    gc = (big_seq.count('G') + big_seq.count('C')) / len(big_seq) * 100
    print(f"GC content: {gc:.2f}%")

In [ ]:
@contextmanager
def temp_fasta(sequences):
    """Create a temporary FASTA file, automatically clean up when done."""
    import tempfile

    fd, path = tempfile.mkstemp(suffix='.fasta')
    try:
        with os.fdopen(fd, 'w') as f:
            for seq_id, seq in sequences.items():
                f.write(f">{seq_id}\n{seq}\n")
        yield path
    finally:
        os.unlink(path)
        print(f"Cleaned up temp file: {path}")


# Temp file exists only inside the with block
test_seqs = {
    "gene1": "ATGCATGCATGC",
    "gene2": "GGCCAATTGGCC",
    "gene3": "TTAACCGGTTAA",
}

with temp_fasta(test_seqs) as fasta_path:
    print(f"Temp file: {fasta_path}")
    with open(fasta_path) as f:
        print(f.read())

## 11. FASTA Processing Pipeline: Combining Patterns

A real-world example combining a context manager with iteration and statistics.

In [ ]:
from dataclasses import dataclass


@dataclass
class FastaRecord:
    id: str
    description: str
    sequence: str

    @property
    def gc_content(self):
        seq = self.sequence.upper()
        return (seq.count('G') + seq.count('C')) / len(seq) * 100


class FastaProcessor:
    """Context manager that parses FASTA and tracks statistics."""

    def __init__(self, filename):
        self.filename = filename
        self.file = None
        self.records_processed = 0
        self.total_bases = 0
        self._start_time = None

    def __enter__(self):
        self.file = open(self.filename, 'r')
        self._start_time = time.perf_counter()
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        if self.file:
            self.file.close()
        elapsed = time.perf_counter() - self._start_time
        print(f"\nProcessing Summary:")
        print(f"  Records: {self.records_processed}")
        print(f"  Total bases: {self.total_bases:,}")
        print(f"  Time: {elapsed:.3f}s")
        return False

    def records(self):
        """Generator yielding FastaRecord objects."""
        current_id = None
        current_desc = ""
        parts = []

        for line in self.file:
            line = line.strip()
            if line.startswith('>'):
                if current_id:
                    seq = ''.join(parts)
                    self.records_processed += 1
                    self.total_bases += len(seq)
                    yield FastaRecord(current_id, current_desc, seq)
                header_parts = line[1:].split(None, 1)
                current_id = header_parts[0]
                current_desc = header_parts[1] if len(header_parts) > 1 else ""
                parts = []
            elif current_id:
                parts.append(line)

        if current_id:
            seq = ''.join(parts)
            self.records_processed += 1
            self.total_bases += len(seq)
            yield FastaRecord(current_id, current_desc, seq)


with FastaProcessor('output_test.fasta') as processor:
    for record in processor.records():
        print(f"{record.id}: {len(record.sequence)} bp, GC={record.gc_content:.1f}%")

## 12. Real-World Pattern: Retry Decorator for Network Calls

In [ ]:
def retry(max_attempts=3, delay=0.5):
    """Decorator factory: retry a function on failure."""
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            last_error = None
            for attempt in range(1, max_attempts + 1):
                try:
                    return func(*args, **kwargs)
                except Exception as e:
                    last_error = e
                    print(f"  Attempt {attempt}/{max_attempts} failed: {e}")
                    if attempt < max_attempts:
                        time.sleep(delay)
            raise last_error
        return wrapper
    return decorator


@retry(max_attempts=3, delay=0.2)
def fetch_sequence(accession):
    """Simulate fetching a sequence from NCBI (unreliable network)."""
    if random.random() < 0.6:
        raise ConnectionError(f"Timeout fetching {accession}")
    return f"ATGCGATCG{'ATCG' * 10}"


try:
    seq = fetch_sequence("NM_007294")
    print(f"Got sequence: {seq[:20]}... ({len(seq)} bp)")
except ConnectionError as e:
    print(f"All attempts failed: {e}")

---

## Exercises

### Exercise 1: `@count_calls` Decorator

Write a decorator `count_calls` that tracks how many times a function has been called.
Store the count as an attribute `call_count` on the wrapper function.

In [ ]:
# Exercise 1: Your solution

# Test:
# @count_calls
# def gc_content(seq):
#     seq = seq.upper()
#     return (seq.count('G') + seq.count('C')) / len(seq) * 100
#
# gc_content("ATGCGATC")
# gc_content("GGCCAATT")
# gc_content("ATATATATA")
# print(f"Called {gc_content.call_count} times")  # 3

### Exercise 2: `@codon_frame` Decorator

Write a decorator that checks whether the input DNA sequence length is
divisible by 3. If not, it trims the sequence from the end to make it
divisible by 3 and prints a warning, then passes the trimmed sequence
to the wrapped function.

In [ ]:
# Exercise 2: Your solution

# Test:
# @codon_frame
# def translate(seq):
#     codons = [seq[i:i+3] for i in range(0, len(seq), 3)]
#     return codons
#
# print(translate("ATGGCTTAG"))   # ['ATG', 'GCT', 'TAG']
# print(translate("ATGGCTTAGC"))  # Warning + ['ATG', 'GCT', 'TAG']

### Exercise 3: FASTA Reader Context Manager

Write a context manager class `FastaReader` that:
- Opens a FASTA file on enter
- Provides a `next_record()` method that returns `(header, sequence)` tuples
- On exit, prints how many records were read and closes the file
- Handles `FileNotFoundError` gracefully in `__enter__`

In [ ]:
# Exercise 3: Your solution

# Test:
# with FastaReader("output_test.fasta") as reader:
#     for header, seq in reader.next_record():
#         print(f"{header}: {len(seq)} bp")

### Exercise 4: `@min_length` Decorator with Parameter

Write a decorator factory `min_length(n)` that raises `ValueError` if
the first string argument to the decorated function has fewer than `n` characters.

In [ ]:
# Exercise 4: Your solution

# Test:
# @min_length(10)
# def find_orfs(seq):
#     return [i for i in range(len(seq) - 2) if seq[i:i+3] == 'ATG']
#
# print(find_orfs("ATGCGATCGATCG"))  # works
# print(find_orfs("ATGCGA"))          # ValueError: sequence too short

### Exercise 5: `@contextmanager` for Quality-Filtered FASTA

Write a context manager using `@contextmanager` called `quality_fasta_writer`
that:
- Takes a filename and a `min_gc` / `max_gc` range
- Provides a `write(seq_id, sequence)` function (via a helper object or closure)
- Only writes sequences whose GC content falls within the range
- On exit, prints how many sequences were written vs. how many were filtered out

In [ ]:
# Exercise 5: Your solution

# Test:
# with quality_fasta_writer("filtered.fasta", min_gc=40, max_gc=60) as writer:
#     writer.write("seq1", "ATGCATGCATGC")  # GC=50% -- written
#     writer.write("seq2", "AATTAATTAATT")  # GC=0% -- filtered
#     writer.write("seq3", "GGCCGGCCGGCC")  # GC=100% -- filtered
#     writer.write("seq4", "ATGCGATCGATC")  # GC=50% -- written

---

## Summary

### Decorators

| Pattern | Use Case |
|---------|----------|
| `@timer` | Benchmark algorithm performance |
| `@memoize` / `@lru_cache` | Cache expensive computations |
| `@validate_sequence` | Check inputs before processing |
| `@repeat(n)` | Run analysis multiple times |
| `@retry(n)` | Handle unreliable network calls |

### Context Managers

| Pattern | Use Case |
|---------|----------|
| `FastaWriter` | Safe FASTA file writing |
| `FastaProcessor` | Parse + track statistics |
| `temp_fasta()` | Auto-cleanup temporary files |
| `timed_section()` | Time code blocks |

**Key rules:**
- Always use `@functools.wraps(func)` in your decorators
- Context managers guarantee cleanup even when exceptions occur
- Use `@contextmanager` for simple cases, classes for complex ones

In [ ]:
# Cleanup
import os
for f in ['output_test.fasta']:
    if os.path.exists(f):
        os.remove(f)
        print(f"Cleaned up: {f}")

---

[< Previous: Classes and Object-Oriented Programming](../13_Classes_and_OOP/02_oop.ipynb) | [Tier 1: Python for Bioinformatics Overview](../README.md) | [Next: Decorators + Context Managers: Overview >](01_decorators_and_context_managers.ipynb)

---